In [1]:
from pathlib import Path

import pandas as pd

from inside_rails.source_sqlite import connect_read_only


# Locate the repository root robustly. The investigation may live below
# notebooks/ or studies/, so do not infer the database path from cwd alone.
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (
            (candidate / "docs" / "STUDY_DATABASE_REFERENCE.md").exists()
            and (candidate / "data" / "reference" / "course_locations.csv").exists()
        ):
            return candidate

    raise RuntimeError(
        "Inside Rails repository root could not be located from the current directory."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

# These are the exact governed paths documented for the current investigation.
DATABASE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "database"
    / "releases"
    / "inside_rails_v3.sqlite3"
)
COURSE_REFERENCE = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "course_locations.csv"
)
RACE_VIEW = "view_reconciled_race_occurrences"

assert DATABASE.exists(), f"Accepted Database v3 not found: {DATABASE}"
assert COURSE_REFERENCE.exists(), f"Course reference not found: {COURSE_REFERENCE}"


with connect_read_only(DATABASE) as connection:
    # Fail before analysis if the governed race interface no longer exposes
    # the fields required for this precise course/venue question.
    view_columns = {
        row[1]
        for row in connection.execute(
            f"PRAGMA table_info({RACE_VIEW})"
        ).fetchall()
    }

    required_columns = {
        "raw_date",
        "candidate_course_label",
        "candidate_jurisdiction",
        "physical_venue_name",
    }
    missing_columns = required_columns - view_columns

    assert not missing_columns, (
        "Required governed race fields are missing: "
        f"{sorted(missing_columns)}"
    )

    # Quantify GB physical-venue missingness at race level without loading
    # the 189,043-race view into pandas.
    gb_coverage = pd.read_sql_query(
        f"""
        SELECT
            COUNT(*) AS gb_races,
            SUM(CASE
                WHEN physical_venue_name IS NULL THEN 1
                ELSE 0
            END) AS races_missing_physical_venue,
            COUNT(DISTINCT candidate_course_label) AS gb_candidate_course_identities,
            COUNT(DISTINCT CASE
                WHEN physical_venue_name IS NULL
                THEN candidate_course_label
            END) AS identities_missing_physical_venue
        FROM {RACE_VIEW}
        WHERE candidate_jurisdiction = 'Great Britain'
        """,
        connection,
    )

    # Reduce the affected population to one row per governed GB candidate
    # course identity before comparing it with the permanent reference.
    affected_identities = pd.read_sql_query(
        f"""
        SELECT
            candidate_course_label,
            candidate_jurisdiction,
            COUNT(*) AS races,
            COUNT(DISTINCT raw_date) AS race_dates,
            MIN(raw_date) AS earliest_date,
            MAX(raw_date) AS latest_date
        FROM {RACE_VIEW}
        WHERE candidate_jurisdiction = 'Great Britain'
          AND physical_venue_name IS NULL
        GROUP BY
            candidate_course_label,
            candidate_jurisdiction
        ORDER BY
            races DESC,
            candidate_course_label
        """,
        connection,
    )


# Compare the database residue with the governed reference itself.
# A matched reference row with a blank physical venue is different from
# a missing identity or a populated reference value lost during integration.
course_reference = pd.read_csv(COURSE_REFERENCE)

reference_columns = [
    "candidate_course_label",
    "candidate_jurisdiction",
    "physical_venue_name",
    "iana_timezone",
    "location_validation_status",
]

reference_check = affected_identities.merge(
    course_reference[reference_columns],
    on=["candidate_course_label", "candidate_jurisdiction"],
    how="left",
    validate="one_to_one",
    indicator="reference_match",
)

reference_summary = pd.DataFrame(
    {
        "affected_identities": [len(reference_check)],
        "matched_reference_identities": [
            int((reference_check["reference_match"] == "both").sum())
        ],
        "unmatched_reference_identities": [
            int((reference_check["reference_match"] != "both").sum())
        ],
        "matched_but_reference_venue_blank": [
            int(
                (
                    (reference_check["reference_match"] == "both")
                    & reference_check["physical_venue_name"].isna()
                ).sum()
            )
        ],
        "reference_venue_present_but_database_null": [
            int(
                (
                    (reference_check["reference_match"] == "both")
                    & reference_check["physical_venue_name"].notna()
                ).sum()
            )
        ],
    }
)

display(gb_coverage)
display(reference_summary)
display(
    reference_check[
        [
            "candidate_course_label",
            "races",
            "race_dates",
            "earliest_date",
            "latest_date",
            "physical_venue_name",
            "iana_timezone",
            "location_validation_status",
            "reference_match",
        ]
    ]
)

,gb_races,races_missing_physical_venue,gb_candidate_course_identities,identities_missing_physical_venue
0,111634,39398,65,24


,affected_identities,matched_reference_identities,unmatched_reference_identities,matched_but_reference_venue_blank,reference_venue_present_but_database_null
0,24,24,0,24,0


,candidate_course_label,races,race_dates,earliest_date,latest_date,physical_venue_name,iana_timezone,location_validation_status,reference_match
0,Kempton (AW),5046,662,2015-01-07,2026-05-27,NaN,Europe/London,unassigned,both
1,Lingfield (AW),4803,742,2015-01-03,2026-05-12,NaN,Europe/London,unassigned,both
2,Ayr,2325,328,2015-01-02,2026-05-20,NaN,Europe/London,unassigned,both
3,Chepstow,2318,326,2015-01-30,2026-05-21,NaN,Europe/London,unassigned,both
4,Haydock,2269,331,2015-01-17,2026-05-23,NaN,Europe/London,unassigned,both
5,Catterick,2010,287,2015-01-01,2026-05-21,NaN,Europe/London,unassigned,both
6,Ascot,1816,273,2015-01-17,2026-05-09,NaN,Europe/London,unassigned,both
7,Market Rasen,1545,224,2015-01-15,2026-05-08,NaN,Europe/London,unassigned,both
8,Brighton,1484,214,2015-04-21,2025-10-16,NaN,Europe/London,unassigned,both
9,Lingfield,1457,248,2015-01-05,2026-05-26,NaN,Europe/London,unassigned,both
